# 🇩🇪 RKI Germany Surveillance Data

This notebook demonstrates how to access infectious disease surveillance data from Germany's Robert Koch Institute (RKI).

**Data Sources:**
- **COVID-19 Nowcasting** - R estimates and case projections
- **Hospitalizations** - COVID-19 hospital admissions
- **Vaccinations** - COVID-19 vaccination coverage
- **Notifiable Diseases** - 25+ infectious diseases (Measles, Pertussis, etc.)
- **Influenza** - Weekly surveillance reports

**Requirements:**
```bash
pip install pandas matplotlib seaborn requests
```

## 1. Setup and Imports

In [ ]:
import sys
import warnings
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
%matplotlib inline

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Add the scripts directory to path
sys.path.insert(0, '../../scripts')

# Import RKI accessor
from accessors.rki_germany import RKIGermanyAccessor

print("✅ Imports completed successfully!")
print(f"⏰ Current time: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

## 2. Initialize RKI Accessor

In [ ]:
# Initialize the RKI accessor
rki = RKIGermanyAccessor()

print("🇩🇪 RKI Germany Accessor initialized")
print(f"Available data repositories: {list(rki.REPOS.keys())}")

## 3. Explore German Federal States (Bundesländer)

In [ ]:
# List German federal states
states = rki.list_states()
print(f"\n🇩🇪 German Federal States ({len(states)} total):")
print(states.head(10).to_string(index=False))

# Visualize states
fig, ax = plt.subplots(figsize=(10, 8))
y_pos = range(len(states))
ax.barh(y_pos, [1]*len(states))
ax.set_yticks(y_pos)
ax.set_yticklabels(states['state_name'], fontsize=9)
ax.set_xlabel('Count')
ax.set_title('German Federal States (Bundesländer)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Explore Notifiable Diseases

In [ ]:
# List notifiable infectious diseases
diseases = rki.list_notifiable_diseases()
print(f"\n🦠 Notifiable Diseases ({len(diseases)} total):")
print(diseases.head(15).to_string(index=False))

# Count diseases by category
print("\n📊 Sample diseases:")
for idx, row in diseases.head(10).iterrows():
    print(f"  • {row['disease_code']} ({row['disease_name_de']}) - ICD-10: {row['disease_name_en']}")

## 5. Fetch COVID-19 Nowcasting Data

In [ ]:
# Fetch COVID-19 nowcasting data (last 90 days)
end_date = datetime.now()
start_date = end_date - timedelta(days=90)

print(f"📊 Fetching COVID-19 nowcasting data ({start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')})...")

covid_nowcast = rki.get_covid_nowcast(
    date_range=(start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d'))
)

if not covid_nowcast.empty:
    print(f"✅ Retrieved {len(covid_nowcast)} records")
    print(f"\n📋 Columns: {list(covid_nowcast.columns)}")
    print("\n🔍 Sample data:")
    print(covid_nowcast.head())
else:
    print("⚠️ No data retrieved. This may be due to:")
    print("   - Network connectivity issues")
    print("   - Changes in RKI data structure")
    print("   - Git LFS storage (large files)")

## 6. Visualize COVID-19 R Value

In [ ]:
# Plot R value over time
if not covid_nowcast.empty and 'r_value' in covid_nowcast.columns:
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Plot R value
    ax.plot(covid_nowcast['date'], covid_nowcast['r_value'], 
            linewidth=2, color='steelblue', label='7-day R value')
    
    # Add confidence intervals if available
    if 'r_lower_ci' in covid_nowcast.columns and 'r_upper_ci' in covid_nowcast.columns:
        ax.fill_between(covid_nowcast['date'], 
                        covid_nowcast['r_lower_ci'], 
                        covid_nowcast['r_upper_ci'], 
                        alpha=0.3, color='steelblue', label='95% CI')
    
    # Add critical threshold line
    ax.axhline(y=1.0, color='red', linestyle='--', linewidth=2, label='R = 1.0 (threshold)')
    
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('R Value', fontsize=12)
    ax.set_title('COVID-19 Effective Reproduction Number (R) - Germany', fontsize=14)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print("\n📊 R Value Statistics:")
    print(f"  Mean R: {covid_nowcast['r_value'].mean():.2f}")
    print(f"  Current R: {covid_nowcast['r_value'].iloc[-1]:.2f}")
    print(f"  Days above threshold (R>1): {(covid_nowcast['r_value'] > 1).sum()}")
    print(f"  Days below threshold (R<1): {(covid_nowcast['r_value'] < 1).sum()}")

## 7. Fetch COVID-19 Hospitalization Data

In [ ]:
# Fetch hospitalization data for selected states
selected_states = ['DE-BY', 'DE-BE', 'DE-NW']  # Bavaria, Berlin, North Rhine-Westphalia

print(f"🏥 Fetching COVID-19 hospitalization data for {len(selected_states)} states...")

hosp_data = rki.get_covid_hospitalizations(
    states=selected_states,
    date_range=(start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d'))
)

if not hosp_data.empty:
    print(f"✅ Retrieved {len(hosp_data)} hospitalization records")
    print("\n🔍 Sample data:")
    print(hosp_data.head())
else:
    print("⚠️ No hospitalization data retrieved")

## 8. Compare Hospitalizations Across States

In [ ]:
# Visualize hospitalizations by state
if not hosp_data.empty and 'Bundesland' in hosp_data.columns:
    # Group by state and date
    hosp_by_state = hosp_data.groupby(['Datum', 'Bundesland'])['7T_Hospitalisierung_Faelle'].sum().reset_index()
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    for state in hosp_by_state['Bundesland'].unique():
        state_data = hosp_by_state[hosp_by_state['Bundesland'] == state]
        ax.plot(state_data['Datum'], state_data['7T_Hospitalisierung_Faelle'], 
                linewidth=2, label=state, marker='o', markersize=3)
    
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('7-day Hospitalization Cases', fontsize=12)
    ax.set_title('COVID-19 Hospitalizations by State - Germany', fontsize=14)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    # Summary by state
    print("\n📊 Total Hospitalizations by State:")
    state_totals = hosp_data.groupby('Bundesland')['7T_Hospitalisierung_Faelle'].sum().sort_values(ascending=False)
    for state, total in state_totals.items():
        print(f"  {state}: {total:,.0f} cases")

## 9. Fetch Vaccination Data

In [ ]:
# Fetch vaccination data
print("💉 Fetching COVID-19 vaccination data...")

vacc_data = rki.get_covid_vaccinations(states=['DE-BY', 'DE-BE'])

if not vacc_data.empty:
    print(f"✅ Retrieved {len(vacc_data)} vaccination records")
    print("\n🔍 Sample data:")
    print(vacc_data.head())
    
    # Show available columns
    print(f"\n📋 Available columns: {list(vacc_data.columns)}")
else:
    print("⚠️ No vaccination data retrieved")

## 10. Example: Query Specific Notifiable Disease

In [ ]:
# Query measles data (placeholder - requires SurvStat@RKI API)
print("🔍 Querying Measles data...")

measles_data = rki.get_notifiable_disease(
    disease="Measles",
    years=[2022, 2023],
    states=['DE-BY', 'DE-BE', 'DE-NW']
)

if not measles_data.empty:
    print(f"✅ Retrieved {len(measles_data)} records")
    print("\n🔍 Sample data structure:")
    print(measles_data.head())
else:
    print("⚠️ Note: Notifiable disease data requires SurvStat@RKI 2.0 access")
    print("   Visit: https://survstat.rki.de/")
    
print("\n📋 Data structure includes:")
print("  • disease_code: Disease identifier")
print("  • year: Reporting year")
print("  • state_code: Federal state")
print("  • cases: Number of reported cases")
print("  • incidence_per_100k: Incidence rate")

## 11. Summary Statistics Dashboard

In [ ]:
# Create a summary dashboard
print("="*60)
print("📊 RKI GERMANY SURVEILLANCE DATA SUMMARY")
print("="*60)

print("\n🇩🇪 Coverage:")
print(f"  • Federal States: {len(rki.STATES)}")
print(f"  • Notifiable Diseases: {len(rki.NOTIFIABLE_DISEASES)}")
print(f"  • Age Groups: {len(rki.AGE_GROUPS)}")

print("\n📁 Available Data Sources:")
for key, repo in rki.REPOS.items():
    print(f"  • {key}: {repo}")

if not covid_nowcast.empty:
    print("\n📈 COVID-19 Latest Data:")
    print(f"  • Analysis period: {covid_nowcast['date'].min().strftime('%Y-%m-%d')} to {covid_nowcast['date'].max().strftime('%Y-%m-%d')}")
    print(f"  • Current R value: {covid_nowcast['r_value'].iloc[-1]:.2f}")
    print(f"  • Data points: {len(covid_nowcast)}")

print("\n💡 Usage Examples:")
print("  rki = RKIGermanyAccessor()")
print("  covid = rki.get_covid_nowcast(date_range=('2024-01-01', '2024-12-31'))")
print("  hosp = rki.get_covid_hospitalizations(states=['DE-BY', 'DE-BE'])")
print("  vacc = rki.get_covid_vaccinations()")
print("  measles = rki.get_notifiable_disease('Measles', years=[2023])")

print("\n✅ RKI Germany accessor ready to use!")
print("="*60)

## 12. Data Export

In [ ]:
# Export data to CSV (optional)
export_path = "./rki_export/"
import os
os.makedirs(export_path, exist_ok=True)

if not covid_nowcast.empty:
    covid_nowcast.to_csv(f"{export_path}covid_nowcast.csv", index=False)
    print(f"✅ Exported COVID nowcast data to {export_path}covid_nowcast.csv")

if not hosp_data.empty:
    hosp_data.to_csv(f"{export_path}covid_hospitalizations.csv", index=False)
    print(f"✅ Exported hospitalization data to {export_path}covid_hospitalizations.csv")

print(f"\n📁 All exports saved to: {os.path.abspath(export_path)}")